In [ ]:
import time
start_time = time.time()

In [ ]:
import sys
import os

# Detect if the environment is Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Adjust this path to the location of your project folder in Google Drive
    GOOGLE_DRIVE_PATH = "/content/drive/MyDrive/Semantics"

    if os.path.exists(GOOGLE_DRIVE_PATH):
        os.chdir(GOOGLE_DRIVE_PATH)
        sys.path.append(GOOGLE_DRIVE_PATH)
        print(f"Successfully changed working directory to: {os.getcwd()}")
    else:
        print(f"Error: Google Drive path '{GOOGLE_DRIVE_PATH}' does not exist.")
else:
    print("Running in local environment; skipping Google Drive mount.")


In [ ]:
# Install dependencies required by Docling and Continued Pretraining (CPT/DAPT)
!pip install -q docling python-dotenv transformers accelerate torch boto3 tiktoken hf-xet pydrive2 bert-score wandb tqdm
!pip install -q sentence-transformers faiss-cpu rank-bm25 hdbscan nltk
!pip install -U "torchao>=0.16.0"

In [ ]:
import os
from dotenv import load_dotenv
from lib.utils import PipelineConfig

# Instantiate and validate pipeline config
cfg = PipelineConfig()
cfg.validate()
cfg.ensure_dirs()

In [ ]:
from lib.utils import FunctionProfiler
from lib.s1_build_corpus import run_corpus_builder
from lib.s2_pretokenize import run_pretokenization
from lib.s3_dapt import run_dapt_pipeline
from lib.s3_dapt.evaluation.eval_runner import run_inference_and_log_failures
from lib.s4_rad_prep import run_rad_prep_pipeline
from lib.s5_clustering import run_clustering_pipeline
from lib.s6_teacher_benchmarking import run_teacher_benchmarking


In [ ]:
# Colab GPU telemetry. This records instrumentation only; DAPT inputs and settings are unchanged.
from datetime import datetime
from pathlib import Path
import shutil
import subprocess
import torch

ENABLE_GPU_MONITOR = True
profile_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
profile_output_dir = Path("/content/dapt_profiler") if IN_COLAB else Path("logs/profiler")
profile_output_dir.mkdir(parents=True, exist_ok=True)
gpu_metrics_local_path = profile_output_dir / f"gpu_metrics_{profile_timestamp}.csv"
gpu_metrics_project_path = Path("logs/profiler") / gpu_metrics_local_path.name
gpu_monitor = None
gpu_monitor_handle = None

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
if ENABLE_GPU_MONITOR and torch.cuda.is_available():
    gpu_monitor_handle = gpu_metrics_local_path.open("w", encoding="utf-8")
    gpu_monitor = subprocess.Popen(
        ["nvidia-smi", "--query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used,memory.total,power.draw,clocks.sm", "--format=csv,noheader,nounits", "-l", "1"],
        stdout=gpu_monitor_handle,
        stderr=subprocess.STDOUT,
    )
    print(f"Recording one GPU sample per second to {gpu_metrics_local_path}")


In [ ]:
cfg.optimizer.train_batch_size = 8
cfg.optimizer.eval_batch_size = 32
cfg.model.gradient_checkpointing = True
cfg.model.peft_dapt = True
probe_seq_len = 128
probe_batch_size = 128

# RAD Prep (Step 0.3) Configurations
rad_mode = "full"
cfg.rad.embed_batch_size = 256
cfg.rad.teacher_batch_size = 16
cfg.rad.trace_min_tokens = 15
cfg.rad.teacher_backend = "bedrock"
cfg.rad.teacher_model_name = "apac.amazon.nova-pro-v1:0"
os.environ["AWS_ACCESS_KEY_ID"] = ""                # update before running
os.environ["AWS_SECRET_ACCESS_KEY"] = ""            # update before running
os.environ["AWS_DEFAULT_REGION"] = "ap-south-1"     # Or your specific region
os.environ["HF_TOKEN"] = ""                         # update before running

# Step 5 Corpus Engineering & Micro-Clustering Configurations
cfg.clustering.pca_components = 20
cfg.clustering.hdbscan_min_cluster_size = 6
cfg.clustering.hdbscan_min_samples = 1
cfg.clustering.noise_assignment = "nearest"



# Step 6 Teacher Benchmarking Configurations
cfg.benchmarking.candidate_teachers = ["apac.amazon.nova-pro-v1:0"]  # Candidate teacher(s)
cfg.benchmarking.teacher_backend = "bedrock"                        # Teacher generation backend
cfg.benchmarking.judge_backend = "bedrock"                          # LLM Judge backend
cfg.benchmarking.judge_model_name = "apac.amazon.nova-pro-v1:0"     # LLM Judge model name
cfg.benchmarking.eval_sample_size = 10                              # Max eval samples per cluster
cfg.benchmarking.min_eval_samples = 2                               # Min eval samples per cluster

cfg.probes.perplexity_max_seq_len = probe_seq_len
cfg.probes.perplexity_batch_size = probe_batch_size
cfg.probes.qa_batch_size = probe_batch_size
cfg.probes.qa_max_seq_len = probe_seq_len
cfg.probes.cloze_gen_batch_size = probe_batch_size
cfg.probes.cloze_max_seq_len = probe_seq_len
cfg.probes.concept_gen_batch_size = probe_batch_size
cfg.probes.concept_bertscore_batch_size = probe_batch_size
cfg.probes.concept_max_seq_len = probe_seq_len

# Set pilot run evaluation intervals explicitly to test convergence gates
cfg.corpus.eval_interval_tokens = 2000000
cfg.corpus.slow_eval_interval_tokens = 4000000

# In Colab, redirect checkpoints and datasets to local NVMe (/content/) to avoid slow Drive I/O.
if IN_COLAB:
    from pathlib import Path
    import shutil
    
    local_ckpt_dir = Path("/content/dapt_checkpoints")
    local_ckpt_dir.mkdir(parents=True, exist_ok=True)
    cfg.model.checkpoint_dir = local_ckpt_dir
    
    # Fast NVMe Staging Paths for RAD Prep
    cfg.rad.chunks_path = Path("/content/data/rad_prep/chunks.jsonl")
    cfg.rad.index_dir = Path("/content/data/rad_prep/index")
    cfg.rad.traces_dir = Path("/content/data/rad_prep/traces")
    
    # Copy pretokenized training tokens to local NVMe to avoid slow Drive I/O (mmap)
    local_train_tokens = Path("/content/train_tokens.npy")
    if cfg.data.pretokenized_bin_path.exists():
        print("Copying pretokenized train tokens to local /content/")
        shutil.copy2(cfg.data.pretokenized_bin_path, local_train_tokens)
        cfg.data.pretokenized_bin_path = local_train_tokens
        
    # Copy validation perplexity tokens to local NVMe
    local_ppl_tokens = Path("/content/ppl_validation_tokens.npy")
    if cfg.data.ppl_corpus_path.exists():
        print("Copying ppl validation tokens to local /content/")
        shutil.copy2(cfg.data.ppl_corpus_path, local_ppl_tokens)
        cfg.data.ppl_corpus_path = local_ppl_tokens

In [ ]:
with FunctionProfiler(module_path_filter='/content/', min_pct=1.0) as prof:
    # print("Initializing corpus building pipeline using Docling parser")
    # run_corpus_builder(cfg)
    # print("Initializing offline pre-tokenization step")
    # run_pretokenization(cfg)
    # print(f"Initializing DAPT Continued Pretraining on model: {cfg.model.base_model_name}")
    # run_dapt_pipeline(cfg)
    # rad_mode = "full"
    print(f"Initializing Retrieval-Augmented Distillation Preparation (RAD Prep) in mode: {rad_mode}")
    run_rad_prep_pipeline(cfg, rad_mode)
    # print("Initializing Corpus Engineering & Micro-Clustering (Step 5)")
    # run_clustering_pipeline(cfg)
    # print("Initializing Teacher Benchmarking (Phase 2, Step 2.1)")
    # run_teacher_benchmarking(cfg)

In [ ]:
# Sync RAD Prep generated artifacts from local NVMe (/content/data/rad_prep) back to Google Drive
if IN_COLAB and Path("/content/data/rad_prep").exists():
    drive_rad_dir = Path("/content/drive/MyDrive/Semantics/data/rad_prep")
    drive_rad_dir.mkdir(parents=True, exist_ok=True)
    print("Syncing RAD Prep artifacts from local NVMe SSD to Google Drive...")
    shutil.copytree("/content/data/rad_prep", drive_rad_dir, dirs_exist_ok=True)
    print("Sync complete!")

# Sync Step 5 Clustering artifacts from local NVMe (/content/data/clustering) back to Google Drive
if IN_COLAB and Path("/content/data/clustering").exists():
    drive_clustering_dir = Path("/content/drive/MyDrive/Semantics/data/clustering")
    drive_clustering_dir.mkdir(parents=True, exist_ok=True)
    print("Syncing Step 5 Clustering artifacts from local NVMe SSD to Google Drive...")
    shutil.copytree("/content/data/clustering", drive_clustering_dir, dirs_exist_ok=True)
    print("Clustering sync complete!")

# Sync Step 6 Benchmarking artifacts from local NVMe (/content/data/benchmarking) back to Google Drive
if IN_COLAB and Path("/content/data/benchmarking").exists():
    drive_benchmarking_dir = Path("/content/drive/MyDrive/Semantics/data/benchmarking")
    drive_benchmarking_dir.mkdir(parents=True, exist_ok=True)
    print("Syncing Step 6 Benchmarking artifacts from local NVMe SSD to Google Drive...")
    shutil.copytree("/content/data/benchmarking", drive_benchmarking_dir, dirs_exist_ok=True)
    print("Benchmarking sync complete!")

# Sync logs/pipeline.log from local environment to Google Drive
if IN_COLAB:
    from lib.utils import flush_loggers
    flush_loggers()
    drive_logs_dir = Path("/content/drive/MyDrive/Semantics/logs")
    drive_logs_dir.mkdir(parents=True, exist_ok=True)
    if Path("logs/pipeline.log").exists():
        shutil.copy2("logs/pipeline.log", drive_logs_dir / "pipeline.log")
        print("Successfully synced logs/pipeline.log to Google Drive!")


In [ ]:
# Stop telemetry and persist it to Drive/project storage after DAPT has finished.
if gpu_monitor is not None:
    gpu_monitor.terminate()
    try:
        gpu_monitor.wait(timeout=5)
    except subprocess.TimeoutExpired:
        gpu_monitor.kill()
if gpu_monitor_handle is not None:
    gpu_monitor_handle.close()

if gpu_metrics_local_path.exists() and profile_output_dir != Path("logs/profiler"):
    gpu_metrics_project_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(gpu_metrics_local_path, gpu_metrics_project_path)
    gpu_metrics_path = gpu_metrics_project_path
else:
    gpu_metrics_path = gpu_metrics_local_path

if torch.cuda.is_available():
    gib = float(2**30)
    memory_summary = {
        "allocated_gib": round(torch.cuda.memory_allocated() / gib, 3),
        "reserved_gib": round(torch.cuda.memory_reserved() / gib, 3),
        "peak_allocated_gib": round(torch.cuda.max_memory_allocated() / gib, 3),
        "peak_reserved_gib": round(torch.cuda.max_memory_reserved() / gib, 3),
    }
    print(f"CUDA memory summary: {memory_summary}")
print(f"GPU telemetry saved to {gpu_metrics_path}")


In [ ]:
import os
from datetime import datetime

# Ensure logs/profiler folder exists
os.makedirs("logs/profiler", exist_ok=True)

# Generate timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = f"logs/profiler/profiler_{timestamp}.log"

# Print to notebook output
prof.print_tree()

# Write to log file
with open(log_path, "w", encoding="utf-8") as f:
    prof.print_tree(file=f)

print(f"\nProfiler output saved to {log_path}")

In [ ]:
end_time = time.time()
diff = end_time - start_time
print(f"Total time taken for the notebook: {diff}")

In [ ]:
from google.colab import runtime
runtime.unassign()